# 第三轮模型：允许时间平移的重新评测
本笔记本只读取 Google Drive 中已有的 `MyDrive/Parrot/trial3_results/plastic/model.pt` 和 `recurrent/model.pt`，不重新训练。先检查模型是否在最后一个外界有声输入帧结束前抢话；通过后，在所有可能的后续起点中找逐帧误差最小的位置。主要指标直接把模型输出与**原始输入片段**比较，保留输入中的频带、持续时间和停顿；另报告与训练时规定的规范目标输出比较的诊断指标。输出起点可平移，但顺序和长度不能随意拉伸。

运行步骤：① 打开 GPU 或 CPU 运行时；② 获取 GitHub 代码；③ 挂载 Drive；④ 运行重新评测；⑤ 将 `MyDrive/Parrot/trial3_alignment.zip` 回传。


In [ ]:
from pathlib import Path
import subprocess
repo = Path('/content/Parrot')
if not repo.exists():
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/DoctorAwe/Parrot.git', str(repo)], check=True)
else:
    subprocess.run(['git', '-C', str(repo), 'pull', '--ff-only'], check=True)
assert (repo / 'trial3_align_eval.py').is_file()
print('代码版本:', subprocess.check_output(['git', '-C', str(repo), 'rev-parse', '--short', 'HEAD'], text=True).strip())
%cd /content/Parrot


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
root = Path('/content/drive/MyDrive/Parrot')
source = root / 'trial3_results'
output = root / 'trial3_alignment'
for variant in ('plastic', 'recurrent'):
    assert (source / variant / 'model.pt').is_file(), f'缺少 {variant} 模型权重'
print('已找到两组模型权重')


In [ ]:
!python trial3_align_eval.py --input {source} --output {output} --count 200


In [ ]:
import json
report = json.loads((output / 'aligned_metrics.json').read_text())
for variant in ('plastic', 'recurrent'):
    print(variant, report['variants'][variant]['0.15']['overall'])
import shutil
archive = shutil.make_archive(str(root / 'trial3_alignment'), 'zip', root_dir=str(root), base_dir='trial3_alignment')
print('请回传:', archive)
